# EAHN — Kaggle Training Notebook (Final Clean Version)

Run every cell **top to bottom**. No inline patches — all fixes are in the repo.

| Cell | Purpose | Gate before next |
|------|---------|-----------------|
| 1 | Auto-detect paths + config | DATA_ROOT printed correctly |
| 2 | File count check | All 5 methods + real > 0 |
| 3 | Clean previous run | — |
| 4 | Clone repo from GitHub | Repo contents printed |
| 5 | Install deps + import check | Prints ALL IMPORTS OK |
| 6 | Dataset loader verification | Prints VERIFICATION PASSED |
| 7  | Train + Evaluate              | Watch AUC-ROC rise above 0.5      |
| 7b | Collapse diagnostic check     | No collapse detected               |
| 7c | Face-alignment diagnostic     | Paste table output + 3 PNGs to me |
| 8 | Metrics table | — |
| 9 | Detection graphs | — |
| 10 | Heatmap videos | — |
| 11 | Zip for download | — |


---
## Cell 1 — Auto-detect paths + configuration

In [ ]:
import os, glob

# ── Auto-detect FF++ data root ───────────────────────────────────────────────
def find_ffpp_root(base="/kaggle/input", max_depth=6):
    for root, dirs, _ in os.walk(base):
        depth = root.replace(base, "").count(os.sep)
        if depth > max_depth:
            dirs.clear()
            continue
        if (os.path.isdir(os.path.join(root, "original_sequences")) and
                os.path.isdir(os.path.join(root, "manipulated_sequences"))):
            return root
    return None

DATA_ROOT  = find_ffpp_root()
REPO_URL   = "https://github.com/umardrazbhatti/EahnCode.git"
REPO_DIR   = "/kaggle/working/EahnCode"
OUTPUT_DIR = "/kaggle/working/outputs"
CACHE_DIR  = "/kaggle/working/.face_cache"

# ── Set number of training epochs ────────────────────────────────────────────
# Use 2 for a quick smoke-test; 10+ for real results
EPOCHS     = 2   # phase7: 2-epoch trial first; change to 10 after validating fixes
BATCH_SIZE = 4

if DATA_ROOT is None:
    print("ERROR: FF++ root not found. Listing /kaggle/input for diagnosis:")
    for e in sorted(glob.glob("/kaggle/input/*/*")):
        print(f"  {e}")
    raise SystemExit("Set DATA_ROOT manually above after checking the listing.")

print("=" * 55)
print(f"DATA_ROOT  : {DATA_ROOT}")
print(f"REPO_DIR   : {REPO_DIR}")
print(f"OUTPUT_DIR : {OUTPUT_DIR}")
print(f"EPOCHS     : {EPOCHS}")
print(f"BATCH_SIZE : {BATCH_SIZE}")
print("=" * 55)
print("If DATA_ROOT looks wrong, set it manually and re-run only this cell.")


---
## Cell 2 — File count sanity check
Every row must show **OK** before continuing.

In [ ]:
import glob, os

METHODS = ["Deepfakes", "Face2Face", "FaceShifter", "FaceSwap", "NeuralTextures"]

real_dir  = os.path.join(DATA_ROOT, "original_sequences", "youtube", "c23", "videos")
real_vids = glob.glob(os.path.join(real_dir, "*.mp4"))
status    = "OK" if len(real_vids) > 0 else "MISSING"
print(f"[{status:^7}]  {'original (real)':<22}  {len(real_vids):>5} videos")

total_fake, all_ok = 0, len(real_vids) > 0
for method in METHODS:
    d      = os.path.join(DATA_ROOT, "manipulated_sequences", method, "c23", "videos")
    count  = len(glob.glob(os.path.join(d, "*.mp4")))
    total_fake += count
    status = "OK" if count > 0 else "MISSING"
    if count == 0: all_ok = False
    print(f"[{status:^7}]  {method:<22}  {count:>5} videos")

print(f"\nTotal real: {len(real_vids)}  |  Total fake: {total_fake}  "
      f"|  Combined: {len(real_vids) + total_fake}")

if not all_ok:
    raise SystemExit("One or more directories missing. Fix DATA_ROOT in Cell 1.")
print("\nFile counts OK — proceed to Cell 3.")


---
## Cell 3 — Clean previous run
Deletes old clone, outputs and cache. Dataset under `/kaggle/input/` is never touched.

In [ ]:
import shutil, os

to_remove = [REPO_DIR, OUTPUT_DIR, "/kaggle/working/eahn_outputs.zip"]
for path in to_remove:
    if os.path.isdir(path):
        shutil.rmtree(path)
        print(f"Removed dir  : {path}")
    elif os.path.isfile(path):
        os.remove(path)
        print(f"Removed file : {path}")
    else:
        print(f"Not present  : {path}")

os.makedirs(OUTPUT_DIR, exist_ok=True)

fc = "/kaggle/working/.face_cache"
if os.path.isdir(fc):
    n_cached = sum(len(files) for _, _, files in os.walk(fc))
    print(f"[face_cache] preserved: {n_cached} cached files at {fc}")
else:
    os.makedirs(fc, exist_ok=True)
    print(f"[face_cache] empty cache created at {fc} (first run will be slow)")

print()
print("Clean. Proceed to Cell 4.")

---
## Cell 4 — Clone repo from GitHub
⚠️ **Internet must be ON**: Kaggle right panel → Settings → Internet → On.

In [ ]:
import os, sys, subprocess

ret = os.system(f"git clone {REPO_URL} {REPO_DIR}")

if ret != 0 or not os.path.isdir(REPO_DIR):
    raise SystemExit(
        "Clone failed.\n"
        "Fix: Enable Internet in Kaggle Settings (right panel → Internet → On).\n"
        "Then re-run this cell."
    )

sys.path.insert(0, REPO_DIR)
os.chdir(REPO_DIR)

branch = subprocess.check_output(["git", "branch", "--show-current"], cwd=REPO_DIR).decode().strip()
commit = subprocess.check_output(["git", "log", "--oneline", "-1"],   cwd=REPO_DIR).decode().strip()
print(f"\nClone successful.")
print(f"  Branch : {branch}")
print(f"  Commit : {commit}")
print("\nRepo contents:")
for f in sorted(os.listdir(REPO_DIR)):
    print(f"  {f}")


---
## Cell 5 — Install dependencies + full import chain check
Must print **ALL IMPORTS OK** before continuing.

In [ ]:
import subprocess, sys, importlib, os

# ── Ensure REPO_DIR is first on sys.path and CWD ─────────────────────────────
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
os.chdir(REPO_DIR)

# ── Flush stale module cache (guards against re-running without restarting kernel)
stale = [k for k in list(sys.modules) if any(k.startswith(p) for p in
         ["config", "data", "models", "losses", "xai", "metrics", "utils", "scripts"])]
for k in stale:
    del sys.modules[k]

print("Installing requirements...")
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r",
     f"{REPO_DIR}/requirements.txt"],
    check=True
)
print("Installation complete.\n")

# ── Key package versions ─────────────────────────────────────────────────────
pkg_map = {"torch":"torch","torchvision":"torchvision","timm":"timm",
           "sklearn":"sklearn","cv2":"cv2","PIL":"PIL","tqdm":"tqdm"}
for name, mod_name in pkg_map.items():
    try:
        m = importlib.import_module(mod_name)
        print(f"  OK      {name:<15} {getattr(m, '__version__', '?')}")
    except ImportError:
        print(f"  MISSING {name}")

# ── Full import chain ─────────────────────────────────────────────────────────
print("\nChecking full import chain...")
try:
    from config import EAHNConfig                            # noqa
    from data.datasets import DeepfakeDataset                # noqa
    from data.collate import deepfake_collate_fn             # noqa
    from data.transforms import get_transforms               # noqa
    from data.face_align import FaceAligner                  # noqa
    from data.synthetic_generator import SyntheticDataGenerator  # noqa
    from models.eahn import EAHN                             # noqa
    from losses.classification import ClassificationLoss     # noqa
    from losses.explanation import ExplanationLoss           # noqa
    from losses.temporal import TemporalConsistencyLoss      # noqa
    from metrics.detection import DetectionMetrics           # noqa
    from scripts.train_real import main as train_main        # noqa
    from scripts.evaluate import run_evaluation              # noqa
    from scripts.dashboard import show_dashboard             # noqa
    print("\nALL IMPORTS OK — proceed to Cell 6.")
except ImportError as e:
    raise SystemExit(
        f"\nIMPORT FAILED: {e}\n"
        "The repo fix (Claude Code prompt) has not been pushed yet, or the "
        "clone in Cell 4 pulled an old commit.\n"
        "Action: re-run from Cell 3 to get a fresh clone."
    )

# ── Verify config params ──────────────────────────────────────────────────────
config = EAHNConfig()
assert hasattr(config, 'attn_temp_init'),        "attn_temp_init missing from EAHNConfig"
assert hasattr(config, 'attn_diversity_weight'), "attn_diversity_weight missing from EAHNConfig"
assert config.gamma == 0.1,             f"gamma must be 0.1, got {config.gamma}"
assert config.cls_loss_type == "focal", f"cls_loss_type must be 'focal', got {config.cls_loss_type!r}"
assert config.cls_dropout_p == 0.0,    f"cls_dropout_p must be 0.0, got {config.cls_dropout_p}"
assert config.batch_size    == 4,      f"batch_size must be 4, got {config.batch_size}"
assert config.grad_accum_steps == 4,   f"grad_accum_steps must be 4, got {config.grad_accum_steps}"
assert getattr(config, 'use_amp', None) is True, "use_amp must be True"
assert getattr(config, 'grad_checkpoint', None) is True, "grad_checkpoint must be True"
assert getattr(config, 'focal_alpha', None) == 1.0, "focal_alpha missing or != 1.0"
assert getattr(config, 'focal_gamma', None) == 1.0,  "focal_gamma missing or != 1.0"
assert hasattr(config, "label_smoothing"),    "label_smoothing missing from EAHNConfig"
assert hasattr(config, "max_per_class"),       "max_per_class missing from EAHNConfig"

print("Config verified:")
print(f"  gamma            = {config.gamma}")
print(f"  attn_temp_init   = {config.attn_temp_init}")
print(f"  diversity_w      = {config.attn_diversity_weight}")
print(f"  cls_loss_type    = {config.cls_loss_type}")
print(f"  cls_dropout_p    = {config.cls_dropout_p}")
print(f"  batch_size       = {config.batch_size}  (effective={config.batch_size * config.grad_accum_steps})  AMP={config.use_amp}  GradCkpt={config.grad_checkpoint}")
print(f"  grad_accum_steps = {config.grad_accum_steps}")
print(f"  use_amp          = {config.use_amp}")
print(f"  grad_checkpoint  = {config.grad_checkpoint}")
print(f"  focal_alpha      = {config.focal_alpha}")
print(f"  focal_gamma      = {config.focal_gamma}")

---
## Cell 6 — Dataset loader verification ✅
Loads one real training batch and asserts both classes are present.
**Do not run Cell 7 unless this prints VERIFICATION PASSED.**


In [ ]:
import os, sys, torch
sys.path.insert(0, REPO_DIR)
os.chdir(REPO_DIR)          # match Cells 4 & 7 — ensures CWD=repo after pip changes sys.path

# Flush module cache so imports are fresh after Cell 5
stale = [k for k in list(sys.modules) if any(k.startswith(p) for p in
         ["data","models","losses","xai","metrics","utils","scripts","config"])]
for k in stale: del sys.modules[k]

from config import EAHNConfig
from data.datasets import DeepfakeDataset
from data.collate  import deepfake_collate_fn
from torch.utils.data import DataLoader, WeightedRandomSampler

# Use num_frames=4 here (quick check only — training uses 16)
_cfg = EAHNConfig(
    data_root    = DATA_ROOT,
    output_dir   = OUTPUT_DIR,
    cache_dir    = CACHE_DIR,
    dataset_name = "ff++",
    num_frames   = 4,
    frame_size   = 224,
    train_split  = 0.8,
    val_split    = 0.1,
    device       = "auto",
    num_workers  = 0,
)

print("Building dataset (train split)...")
ds = DeepfakeDataset(_cfg, mode="train", dataset_type="ff++")
labels_arr = [s["label"] for s in ds.samples]
n_real = labels_arr.count(0); n_fake = labels_arr.count(1)
print(f"  Total samples: {len(ds)}  |  real={n_real}  fake={n_fake}")

weights = [1.0/n_real if l == 0 else 1.0/n_fake for l in labels_arr]
sampler = WeightedRandomSampler(weights, num_samples=len(ds), replacement=True)
loader  = DataLoader(ds, batch_size=8, sampler=sampler,
                     collate_fn=deepfake_collate_fn, num_workers=0)

print("Inspecting 3 sampled batches (WeightedRandomSampler):")
saw_real = saw_fake = False
for i, b in enumerate(loader):
    u = b["label"].unique().tolist()
    r = int((b["label"] == 0).sum()); f = int((b["label"] == 1).sum())
    print(f"  batch {i}: real={r} fake={f} unique={u}")
    if r > 0: saw_real = True
    if f > 0: saw_fake = True
    if i == 2: break

assert saw_real and saw_fake, "Sampler not delivering both classes across 3 batches."
print()
print("="*55)
print("VERIFICATION PASSED — sampler delivers both classes.")
print("Proceed to Cell 7.")
print("="*55)


---
## Cell 7 — Train + Evaluate
Expected: ~4–6 min per epoch on Tesla T4.
After epoch 1 the AUC-ROC should be > 0.5, confirming the model is learning.
The face cache is cleared here first to prevent the T=4 (Cell 6) vs T=16
(training) frame-count collision that caused a RuntimeError in earlier runs.


In [ ]:
import os, sys, shutil
sys.path.insert(0, REPO_DIR)
os.chdir(REPO_DIR)

# ── Clear face cache to prevent T=4 (Cell 6) vs T=16 (training) collision ────
if os.path.isdir(CACHE_DIR):
    shutil.rmtree(CACHE_DIR)
    os.makedirs(CACHE_DIR, exist_ok=True)
    print("Face cache cleared.")

# ── Run full pipeline ─────────────────────────────────────────────
# batch_size=4, B*T=4x16=64 frames/pass, grad_accum_steps=4 -> effective batch=16.
# Fits T4 (14.56 GiB). Do NOT raise batch_size above 4 without grad_checkpoint.
cmd = (
    f'python run_full_pipeline.py'
    f' --data_root    "{DATA_ROOT}"'
    f' --dataset_name ff++'
    f' --output_dir   "{OUTPUT_DIR}"'
    f' --cache_dir    "{CACHE_DIR}"'
    f' --epochs       {EPOCHS}'
    f' --batch_size   4'
    f' --num_workers  2'
    f' --grad_accum_steps 4'
    f' --gamma        0.1'
    f' --attn_temp_init 1.386'
    f' --attn_diversity_weight 2.5'
    f' --cls_dropout_p   0.1'
    f' --cls_loss_type   focal'
    f' --focal_alpha  0.25'
    f' --focal_gamma  2.0'
    f' --use_amp'
    f' --grad_checkpoint'
    f' --eval_after_train'
)
print("Running:", cmd, "
")
ret = os.system(cmd)
if ret != 0:
    print(f"
[ERROR] Pipeline exited with code {ret}. Check logs above.")
else:
    print("
Cell 7 complete.")


---
## Cell 7b — Collapse diagnostic check
Must print **✅ No collapse detected** before proceeding to 10-epoch run.

In [ ]:
import pandas as pd, os
_csv = os.path.join(OUTPUT_DIR, "metrics.csv")
if not os.path.exists(_csv):
    print("metrics.csv not found — did Cell 7 complete?")
else:
    _df = pd.read_csv(_csv, names=["metric","value"], header=0)
    def _get(m):
        row = _df[_df["metric"]==m]["value"]
        return float(row.values[0]) if len(row) else None

    _inter  = _get("inter_sample_cosine_mean")
    _mode   = _get("peak_mode_share")
    _mt_std = _get("m_t_std_mean")

    _warnings = []
    if _inter  is not None and _inter  > 0.95: _warnings.append(f"inter_sample_cosine_mean={_inter:.3f}")
    if _mode   is not None and _mode   > 0.5:  _warnings.append(f"peak_mode_share={_mode:.3f}")
    if _mt_std is not None and _mt_std > 0.13: _warnings.append(f"m_t_std_mean={_mt_std:.3f}")

    if _warnings:
        print("⚠️  EXPLANATION COLLAPSE DETECTED:")
        for w in _warnings: print("   -", w)
        print("Do NOT proceed to 10-epoch run. Diagnose the explanation head first.")
    else:
        print("✅ No collapse detected. Safe to proceed to longer runs.")


---
## Cell 8 — Metrics table

In [ ]:
import os, pandas as pd
from IPython.display import display

csv_path = os.path.join(OUTPUT_DIR, "metrics.csv")
if not os.path.exists(csv_path):
    print("metrics.csv not found. Did Cell 7 complete without errors?")
else:
    df = pd.read_csv(csv_path, header=None, names=["Metric", "Value"])
    df["Value"] = pd.to_numeric(df["Value"], errors="coerce").round(4)

    print("=" * 45)
    print("       EAHN Evaluation Metrics")
    print("=" * 45)
    display(df)

    # ── Quick verdict ─────────────────────────────────────────────────────────
    auc_row = df.loc[df["Metric"] == "auc_roc", "Value"]
    if len(auc_row) and not pd.isna(auc_row.values[0]):
        v = auc_row.values[0]
        if   v >= 0.80: msg = "EXCELLENT — model is performing well."
        elif v >= 0.65: msg = "GOOD — model is learning; more epochs will help."
        elif v >= 0.50: msg = "IMPROVING — learning detected; run more epochs."
        else:           msg = "POOR — near chance; check data pipeline."
        print(f"\nAUC-ROC = {v:.4f}  →  {msg}")
    else:
        print("\nWARNING: AUC-ROC is NaN — evaluation split may be single-class.")
        print("Action: re-run the Claude Code prompt to verify _build_ffpp paths.")
    # Per-class accuracy display (phase5d)
    bal_row  = df[df["Metric"] == "balanced_accuracy_at_optimal"]["Value"]
    real_row = df[df["Metric"] == "real_accuracy"]["Value"]
    fake_row = df[df["Metric"] == "fake_accuracy"]["Value"]
    if len(bal_row):  print(f"Balanced Acc (optimal thr) = {bal_row.values[0]:.4f}")
    if len(real_row): print(f"Real-class accuracy        = {real_row.values[0]:.4f}")
    if len(fake_row): print(f"Fake-class accuracy        = {fake_row.values[0]:.4f}")


    # Display training curve plots (phase6)
    from IPython.display import Image as _Img
    for _fname in ["loss_curves.png", "metric_curves.png", "score_distribution.png"]:
        _fp = os.path.join(OUTPUT_DIR, _fname)
        if os.path.exists(_fp):
            print(f"\n{_fname}:")
            display(_Img(_fp))


---
## Cell 9 — Detection graphs (ROC, PR, confusion matrix, score distribution)

In [ ]:
import os
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

graphs = [
    ("roc_curve.png",         "ROC Curve"),
    ("pr_curve.png",          "Precision-Recall Curve"),
    ("confusion_matrix.png",  "Confusion Matrix"),
    ("score_distribution.png","Score Distribution"),
]
available = [(f, t) for f, t in graphs
             if os.path.exists(os.path.join(OUTPUT_DIR, f))]

if not available:
    print("No graph files found — did Cell 7 produce output?")
else:
    n = len(available)
    fig, axes = plt.subplots(1, n, figsize=(6 * n, 5))
    if n == 1: axes = [axes]
    for ax, (fname, title) in zip(axes, available):
        ax.imshow(mpimg.imread(os.path.join(OUTPUT_DIR, fname)))
        ax.set_title(title, fontsize=12, fontweight="bold")
        ax.axis("off")
    plt.tight_layout()
    plt.show()
    missing = [t for f, t in graphs if (f, t) not in available]
    if missing:
        print(f"Not yet generated: {missing} — scripts/evaluate.py may need updating.")


---
## Cell 10 — Heatmap videos
Displays all 4 XAI methods (intrinsic, gradcam, rollout, shap) for the first sampled video.

In [ ]:
import os
from IPython.display import Video, HTML, display

heatmap_dir = os.path.join(OUTPUT_DIR, "heatmaps")
if not os.path.isdir(heatmap_dir):
    print("No heatmaps directory found. Did Cell 7 complete?")
else:
    methods = ["intrinsic", "gradcam", "rollout", "shap"]
    mp4s    = sorted(f for f in os.listdir(heatmap_dir) if f.endswith(".mp4"))

    # Extract unique video IDs
    vid_ids = sorted(set(
        f.replace(f"_{m}.mp4", "")
        for f in mp4s for m in methods
        if f.endswith(f"_{m}.mp4")
    ))

    if not vid_ids:
        print("No heatmap .mp4 files found in", heatmap_dir)
    else:
        sample = vid_ids[0]
        display(HTML(f"<h3>Sample: {sample} &nbsp;|&nbsp; "
                     f"{len(vid_ids)} total sample(s)</h3>"))
        for method in methods:
            path = os.path.join(heatmap_dir, f"{sample}_{method}.mp4")
            if os.path.exists(path):
                display(HTML(f"<p><b>{method.upper()}</b></p>"))
                display(Video(path, embed=True, width=480))
            else:
                print(f"  [{method}] not found — "
                      f"check xai/{method.replace('shap','shap_explainer')}.py")


---
## Cell 11 — Zip all outputs for download
Download `eahn_outputs.zip` from the Kaggle **Output** tab after this cell.

In [ ]:
import shutil, os

zip_base = "/kaggle/working/eahn_outputs"
shutil.make_archive(zip_base, "zip", OUTPUT_DIR)
size_mb = os.path.getsize(zip_base + ".zip") / 1e6
print(f"Created  : {zip_base}.zip")
print(f"Size     : {size_mb:.1f} MB")
print("Download : Kaggle Output tab → eahn_outputs.zip")
